# Husky 와이어태핑을 통한 오류주입 공격 (FIA via Wire-Tapping)

## 두 ChipWhisperer 장치의 역할 분리 — Lite (통신·프로그래머) + Husky (능동 클럭 글리치 주입)

---

### 🎯 강의 목표

이 노트북은 **ChipWhisperer 입문자를 대상**으로, 기존의 **수동적 와이어태핑(관측 전용)** 시나리오를 **능동적 오류주입 공격(Active Fault Injection)** 으로 확장하는 방법을 단계별로 안내합니다.

- **Lite** : 정상 사용자 역할 (타겟 프로그래밍 + SimpleSerial 통신)
- **Husky** : 공격자 역할 (정밀 클럭 글리치 주입 + 트리거 동기화)

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 다중 장치 연결 (Lite + Husky) | `lite_scope`, `husky_scope` |
| **2단계** | Lite로 타겟 프로그래밍 및 UART 통신 채널 확보 | `target` 객체 |
| **3단계** | Husky 글리치 모듈 설정 (Clock Glitch 모드) | `husky_glitch_setup()` |
| **4단계** | 베이스라인 통신 테스트 | `expected_ret` 확인 |
| **5단계** | 글리치 파라미터 광범위 탐색 (3중 루프) | `cglitch_result_arr` (6단계 분류) |
| **6단계** | 통계 분석으로 최적 파라미터 도출 | `(offset, width)` 최빈값 |
| **7단계** | Bokeh 인터랙티브 시각화 | 파라미터 맵 (결과 코드별 색상) |
| **8단계** | 장비 연결 해제 | 안전한 종료 |

> 💡 **이 노트북의 핵심 차별점**
> - 기존 FA_main.ipynb (단일 장치 능동 FIA) + Wiretapping4SCA.ipynb (다중 장치 수동 SCA) 의 **하이브리드**
> - 실제 공격 시나리오에 더 가까움: 통신은 정상적으로 유지하면서 **클럭 라인만 tap하여 정밀 글리치 주입**



---

## 🔍 시작하기 전에

### 왜 "와이어태핑 + FIA" 인가?

기존 단일 장치 FIA 실습에서는 **측정 장비가 곧 통신 주체**입니다.  
하지만 실제 물리적 공격에서는 공격자가 **이미 동작 중인 시스템의 클럭 라인에만 프로브를 연결**하여 글리치를 주입하고, 정상 통신은 그대로 두는 경우가 많습니다.

본 노트북은 이를 재현합니다:

- **Lite** = "합법적인 사용자 단말" 역할 (평문 전송, 결과 수신)
- **Husky** = "은밀한 공격자" 역할 (트리거 감지 → 정밀 클럭 글리치 주입)

### 하드웨어 연결 구성 (중요!)

```
호스트 PC (Jupyter)
    │
    ├── USB → ChipWhisperer-Lite
    │         ├── UART (TX/RX)  ────────→ Target UART
    │         └── SWD/JTAG      ────────→ Target 프로그래밍
    │
    └── USB → ChipWhisperer-Husky
              ├── Trigger In (D0) ←──── Target GPIO4 / TRIG
              ├── Glitch Out (또는 HS2) ──→ Target CLKIN   ←── Husky가 클럭 + 글리치 제공
              └── (선택) Measure Pos ←──── Target 전력 션트 (추후 SCA 확장용)
```

**핵심 포인트**
- Husky가 **타겟의 시스템 클럭을 직접 공급**하면서 동시에 글리치를 주입합니다.
- Lite는 **UART 통신만** 담당 → 역할 완전 분리.
- 타겟 펌웨어는 FA_main.ipynb와 동일한 FIA 데모 펌웨어 사용 (for-loop 기반 OTP 연산, 글리치로 스킵/변조 가능).

### ⚠️ 안전 주의사항
- 글리치 파라미터를 너무 강하게 (너무 넓은 width) 설정하면 타겟이 **영구 동작 불능** 상태가 될 수 있습니다.
- 처음에는 **좁은 범위**로 테스트하고, 점진적으로 확장하세요.
- Husky의 고해상도 글리치 기능을 활용하면 더 정밀하고 안전한 탐색이 가능합니다.


---

# 📦 1단계 — 라이브러리 임포트 및 다중 장치 연결

> **목표**: 두 장치를 시리얼 넘버 기반으로 명확히 구분하여 연결합니다.


In [ ]:
# 사전 헬퍼 로드 (My_script.ipynb가 있는 경우)
# %run My_script.ipynb

import chipwhisperer as cw
import numpy as np
from tqdm.notebook import trange, tqdm
import time

print("ChipWhisperer 버전:", cw.__version__)

### 1.1 다중 장치 연결 함수 (Wiretapping4SCA.ipynb에서 차용·개선)

In [ ]:
def connect_all_devices():
    """연결된 모든 ChipWhisperer 장치를 시리얼 넘버로 구분하여 연결"""
    device_list = cw.list_devices()
    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다!")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}
    for device in device_list:
        name = device['name'].replace("-", "_")
        sn = device['sn']
        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료 (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패: {e}")
    return scopes

scopes = connect_all_devices()
lite_scope = scopes.get("ChipWhisperer_Lite")
husky_scope = scopes.get("ChipWhisperer_Husky")

if lite_scope is None or husky_scope is None:
    raise RuntimeError("Lite와 Husky가 모두 연결되어 있어야 합니다!")

---

# 🔌 2단계 — Lite를 통한 타겟 보드 연결 및 프로그래밍

> **목표**: Lite가 타겟과 통신하고, 펌웨어를 업로드합니다. (Husky는 아직 관여하지 않음)


In [ ]:
PLATFORM = 'CW308_STM32F3'
SS_VER = 'SS_VER_2_1'

# Lite를 통한 타겟 연결
target_type = cw.targets.SimpleSerial2 if SS_VER == "SS_VER_2_1" else cw.targets.SimpleSerial
target = cw.target(lite_scope, target_type)
print("[✓] Lite를 통해 타겟 연결 완료")

### 펌웨어 빌드 및 프로그래밍 (Lite가 프로그래머 역할)

In [ ]:
import subprocess
import os

FIRMWARE_DIR = "simpleserial-main"  # 실제 경로에 맞게 수정
print("펌웨어 컴파일 중...")

# 컴파일 (FIA용 펌웨어 — for-loop 스킵/변조 데모)
result = subprocess.run(
    ["make", f"PLATFORM={PLATFORM}", "CRYPTO_TARGET=NONE", f"SS_VER={SS_VER}"],
    cwd=FIRMWARE_DIR,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("펌웨어 컴파일 실패")

print("펌웨어 컴파일 완료")

# 프로그래밍 (Lite 사용)
prog = cw.programmers.STM32FProgrammer
lite_scope.default_setup()  # Lite 기본 설정 (UART, gain 등)
cw.program_target(lite_scope, prog, os.path.join(FIRMWARE_DIR, f"simpleserial-base-{PLATFORM}.hex"))
print("[✓] 타겟 프로그래밍 완료 (Lite 경유)")

> **참고**: 프로그래밍 후 Lite의 HS2 클럭 출력은 **사용하지 않습니다**. Husky가 클럭을 공급할 예정입니다.

---

# ⚙️ 3단계 — Husky 글리치 모듈 설정 (Clock Glitch)

> **목표**: Husky를 클럭 글리치 주입기로 구성하고, 타겟 트리거와 동기화합니다.


### Husky 글리치 설정 함수 (FA_main.ipynb 스타일 + Husky 최적화)

In [ ]:
def husky_glitch_setup(scope):
    """
    Husky를 클럭 글리치 모드로 설정
    - 1 sample = 1 clock 정렬 (adc_mul=1)
    - clock_xor 모드 (클럭 글리치에 최적)
    - ext_single 트리거 (타겟 GPIO 트리거 사용)
    """
    scope.glitch.clk_src = "clkgen"          # Husky 내부 클럭 생성기 사용
    scope.glitch.output = "clock_xor"        # 클럭 글리치 (본 실습 추천)
    scope.glitch.trigger_src = "ext_single"  # 외부 트리거 (타겟 GPIO)
    scope.glitch.repeat = 1

    # 고해상도 설정 (Husky 장점)
    scope.clock.adc_mul = 1                  # 1 sample = 1 clock
    scope.clock.clkgen_freq = 7_372_800      # 타겟에 맞는 주파수 (STM32F3 예시)

    # Husky IO 설정
    scope.io.hs2 = "clkgen"                  # Husky가 타겟 클럭 공급
    scope.io.tio1 = "serial_rx"
    scope.io.tio2 = "serial_tx"

    print("[✓] Husky 글리치 모듈 설정 완료")
    print(f"   - output       : {scope.glitch.output}")
    print(f"   - trigger_src  : {scope.glitch.trigger_src}")
    print(f"   - adc_mul      : {scope.clock.adc_mul}")
    print(f"   - clkgen_freq  : {scope.clock.clkgen_freq}")

    return scope

husky_scope = husky_glitch_setup(husky_scope)

### Husky 글리치 파라미터 설명 (복습)

| 파라미터 | 의미 | Husky에서의 특징 |
|----------|------|------------------|
| `ext_offset` | 트리거 후 N 클럭 뒤에 글리치 시작 | 정수 (클럭 단위) |
| `offset` | 1 클럭 내 시작 위상 (0 ~ phase_shift_steps) | **고해상도** (Husky: 수만 단계) |
| `width` | 글리치 펄스 폭 | **고해상도** |
| `phase_shift_steps` | offset/width 최대 단계 수 | Husky에서 매우 큼 → 더 정밀 탐색 가능 |



---

# ✅ 4단계 — 베이스라인 통신 테스트

> **목표**: 글리치 없이 정상 동작을 확인하고, `expected_ret`를 획득합니다.


In [ ]:
def my_fsr_cmd(target, cmd, scmd, data, payload_only=False):
    """SimpleSerial 명령 송수신 헬퍼 (간단 버전)"""
    target.simpleserial_write(cmd, data)
    time.sleep(0.05)
    response = target.simpleserial_read(scmd, 16)  # 16바이트 응답 가정
    return response

# 정상 동작 테스트
print("베이스라인 통신 테스트 중...")
try:
    # FIA 데모 펌웨어에 맞는 명령 (예: 'p'로 평문 전송, 'r'로 결과 수신 등)
    # 실제 펌웨어에 맞게 수정 필요
    resp = my_fsr_cmd(target, 'p', 'r', bytearray([0x00]*16))
    print(f"응답: {resp.hex() if resp else 'None'}")
    expected_ret = resp
    print("[✓] 베이스라인 통신 성공")
except Exception as e:
    print(f"[✗] 통신 실패: {e}")
    expected_ret = None

---

# 🔍 5단계 — 글리치 파라미터 광범위 탐색

> **목표**: `ext_offset`, `offset`, `width`를 체계적으로 탐색하며 결과를 6단계로 분류합니다.


### 결과 분류 함수 (FA_main.ipynb와 동일)

In [ ]:
def classify_glitch_result(response, expected, timeout=False):
    """
    글리치 결과 6단계 분류
    0: freezing (응답 없음)
    1: normal
    2: for-loop skip (응답 길이 짧음)
    3: one faulty byte
    4: few faulty bytes (2~4)
    5: etc (>=5 faulty bytes)
    """
    if timeout or response is None:
        return 0, "freezing"
    if len(response) == 0:
        return 0, "freezing"

    # 길이 비교 (loop skip)
    if len(response) < len(expected):
        return 2, "for-loop skip"

    # 바이트 단위 비교
    faults = sum(1 for a, b in zip(response, expected) if a != b)
    if faults == 0:
        return 1, "normal"
    elif faults == 1:
        return 3, "one faulty byte"
    elif 2 <= faults <= 4:
        return 4, "few faulty bytes"
    else:
        return 5, "etc (>=5 faults)" 

### 글리치 탐색 메인 루프

In [ ]:
# 탐색 범위 설정 (처음에는 좁게! → 점차 확대)
EXT_OFFSET_RANGE = range(0, 50, 5)      # 예시: 좁은 범위
OFFSET_RANGE     = range(0, 200, 20)    # Husky 고해상도 → finer step 가능
WIDTH_RANGE      = range(1, 50, 5)

results = []
print("글리치 파라미터 탐색 시작...")

for ext_off in tqdm(EXT_OFFSET_RANGE, desc="ext_offset"):
    for off in OFFSET_RANGE:
        for wid in WIDTH_RANGE:
            husky_scope.glitch.ext_offset = ext_off
            husky_scope.glitch.offset = off
            husky_scope.glitch.width = wid

            # Husky arm
            husky_scope.arm()

            # 타겟에 명령 전송 (Lite 경유)
            try:
                resp = my_fsr_cmd(target, 'p', 'r', bytearray([0x00]*16))
                code, label = classify_glitch_result(resp, expected_ret)
            except Exception:
                code, label = 0, "freezing"

            results.append([code, ext_off, wid, off, label])

print(f"\n총 {len(results)}개 조합 탐색 완료")

In [ ]:
# 결과를 numpy 배열로 변환
cglitch_result_arr = np.array(results, dtype=object)
print("결과 배열 shape:", cglitch_result_arr.shape)

# 간단한 통계
from collections import Counter
code_counts = Counter(cglitch_result_arr[:, 0])
print("\n결과 코드 분포:")
for code in sorted(code_counts.keys()):
    print(f"  코드 {code}: {code_counts[code]}회")

---

# 📊 6단계 — 통계 분석으로 최적 파라미터 도출

> 성공 사례(코드 2, 3)의 `(offset, width)` 최빈값을 찾아 최적 파라미터 후보로 선정합니다.


In [ ]:
from scipy import stats
import numpy as np

# 성공 사례만 필터링 (2: loop skip, 3: one faulty byte)
success_mask = np.isin(cglitch_result_arr[:, 0], [2, 3])
success_results = cglitch_result_arr[success_mask]

if len(success_results) > 0:
    # offset과 width를 float으로 변환
    offsets = np.array(success_results[:, 3], dtype=float)
    widths  = np.array(success_results[:, 2], dtype=float)

    # 최빈값 (mode)
    mode_offset = stats.mode(offsets, keepdims=True).mode[0]
    mode_width  = stats.mode(widths, keepdims=True).mode[0]

    print(f"✅ 성공 사례 최빈 파라미터")
    print(f"   i_offset (mode): {mode_offset}")
    print(f"   i_width  (mode): {mode_width}")
    print(f"   성공 사례 수: {len(success_results)} / {len(cglitch_result_arr)}")
else:
    print("⚠️ 성공 사례가 없습니다. 탐색 범위를 조정하세요.")

---

# 📈 7단계 — Bokeh으로 파라미터 분포 시각화

> (i_offset, i_width) 평면에 결과 코드를 색상으로 표시하는 인터랙티브 산점도


In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

# 색상 매핑
COLOR_MAP = {
    0: '#888888',  # freezing
    1: '#1f77b4',  # normal
    2: '#2ca02c',  # loop skip ✅
    3: '#d62728',  # one faulty byte ✅
    4: '#ff7f0e',  # few faults
    5: '#9467bd',  # etc
}

LABELS = {
    0: "freezing",
    1: "normal",
    2: "for-loop skip",
    3: "one faulty byte",
    4: "few faulty bytes",
    5: "etc (>=5 faults)"
}

p = figure(
    width=900, height=500,
    title="Glitch Parameter Map (Husky Wire-Tap FIA) — (offset, width)",
    x_axis_label="i_offset (1-clock 내 시작 위상)",
    y_axis_label="i_width (글리치 펄스 폭)",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover",
    active_scroll="wheel_zoom",
    background_fill_color="#fafafa"
)

for code in sorted(set(cglitch_result_arr[:, 0])):
    mask = cglitch_result_arr[:, 0] == code
    if not np.any(mask):
        continue
    src = ColumnDataSource(data=dict(
        x=np.array(cglitch_result_arr[mask, 3], dtype=float),
        y=np.array(cglitch_result_arr[mask, 2], dtype=float),
        ext=np.array(cglitch_result_arr[mask, 1], dtype=float),
        code=[int(code)] * int(mask.sum()),
        label=[LABELS.get(int(code), str(code))] * int(mask.sum())
    ))
    p.scatter('x', 'y', source=src, size=6, alpha=0.6,
              color=COLOR_MAP.get(int(code), '#000000'),
              legend_label=f"[{code}] {LABELS.get(int(code), code)} (n={int(mask.sum())})")

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.legend.label_text_font_size = "9pt"

hover = HoverTool(tooltips=[
    ("result", "@label"),
    ("i_offset", "@x{0}"),
    ("i_width", "@y{0}"),
    ("ext_offset", "@ext{0}")
])
p.add_tools(hover)

show(p)

> **시각화 활용 팁**
> - 범례 클릭으로 특정 결과만 토글
> - Box Zoom으로 군집 영역 확대
> - Hover로 정확한 파라미터 확인
> - `2` (green)와 `3` (red) 군집이 연속적으로 나타나면 신뢰성 높은 공격 파라미터입니다.


---

# 🔚 8단계 — 장비 연결 해제

> **반드시 실행**하여 USB 충돌 방지


In [ ]:
def disconnect_all():
    try:
        target.dis()
        print("[✓] target 연결 해제")
    except:
        pass
    try:
        lite_scope.dis()
        print("[✓] Lite 연결 해제")
    except:
        pass
    try:
        husky_scope.dis()
        print("[✓] Husky 연결 해제")
    except:
        pass

disconnect_all()
print("\n✅ 모든 장치 연결 해제 완료")

---

## 📝 본 노트북 요약

| 단계 | 핵심 내용 | 비고 |
|------|-----------|------|
| 1 | `connect_all_devices()` + 시리얼 넘버 기반 연결 | Lite + Husky 동시 사용 |
| 2 | Lite로 프로그래밍 + `cw.target(lite_scope)` | 통신 전담 |
| 3 | `husky_glitch_setup()` | Husky가 클럭 공급 + 글리치 |
| 4 | Baseline 통신 테스트 | expected_ret 확보 |
| 5 | 3중 루프 + `husky_scope.arm()` + Lite 통신 | 결과 6단계 분류 |
| 6 | `scipy.stats.mode` | 최적 (offset, width) 도출 |
| 7 | Bokeh scatter + 범례 토글 | 시각적 군집 분석 |
| 8 | `disconnect_all()` | 안전한 종료 |

### ✅ 핵심 학습 포인트

1. **역할 분리**의 중요성 — 통신(Lite)과 물리적 공격(Husky) 분리
2. **Husky의 고해상도 글리치** 활용 — 더 정밀한 파라미터 탐색
3. **트리거 기반 정밀 타이밍** — `ext_single` + 타겟 GPIO
4. **현실적 공격 모델** — 정상 통신 유지 + 선택적 클럭 글리치
5. **안전한 탐색 전략** — 좁은 범위 → 통계적 검증 → 시각화

---

**다음 단계 제안**
- 성공한 파라미터로 **반복 공격**하여 DFA 키 복원 실험
- Husky의 Trace 캡처 기능과 결합한 **FIA + SCA 하이브리드 공격**
- Voltage Glitch 확장

*ChipWhisperer 입문 — Husky 와이어태핑 FIA 노트북 끝*
